In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)
from sklearn.inspection import permutation_importance
from scipy.stats import randint, uniform

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

---
## 2. Data Loading

In [ ]:
# Load all splits
X_train = pd.read_csv('../data/X_train(2016-2023).csv', index_col='Date', parse_dates=True)
y_train = pd.read_csv('../data/y_train1(2016-2023).csv', index_col='Date', parse_dates=True).squeeze()

X_val = pd.read_csv('../data/X_val(2024).csv', index_col='Date', parse_dates=True)
y_val = pd.read_csv('../data/y_val(2024).csv', index_col='Date', parse_dates=True).squeeze()

X_test = pd.read_csv('../data/X_test(2025).csv', index_col='Date', parse_dates=True)
y_test = pd.read_csv('../data/y_test(2025).csv', index_col='Date', parse_dates=True).squeeze()

print('=== Dataset Shapes ===')
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}  | y_test:  {y_test.shape}')
print(f'\nFeatures: {X_train.shape[1]}')

---
## 3. Exploratory Data Analysis

### 3.1 Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
splits = [('Train (2016–2023)', y_train), ('Validation (2024)', y_val), ('Test (2025)', y_test)]
colors = ['#2196F3', '#4CAF50', '#FF5722']

for ax, (title, y), color in zip(axes, splits, colors):
    counts = y.value_counts().sort_index()
    bars = ax.bar(['Down (0)', 'Up (1)'], counts.values,
                  color=[color + '99', color], edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts.values):
        pct = count / len(y) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, counts.max() * 1.25)

plt.suptitle('XLE Next-Day Direction', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Macro Feature Distributions (Training Set)

In [ ]:
macro_cols = [c for c in X_train.columns if any(m in c for m in [
    'VIX', 'Oil_Volatility', 'Crude_Oil', 'Natural_Gas', 'Dollar', 'SP500', 'Nikkei', 'FTSE'
])]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(macro_cols):
    ax = axes[i]
    data_up   = X_train.loc[y_train == 1, col]
    data_down = X_train.loc[y_train == 0, col]
    ax.hist(data_down, bins=50, alpha=0.6, color='#E53935', label='Down', density=True)
    ax.hist(data_up,   bins=50, alpha=0.6, color='#43A047', label='Up',   density=True)
    ax.set_title(col.replace('_Log_Return', '').replace('_', ' '), fontsize=9, fontweight='bold')
    ax.set_xlabel('Scaled Log Return')
    ax.legend(fontsize=8)

plt.suptitle('Macro Feature Distributions Direction (Train Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Macro Feature Correlation Heatmap

In [ ]:
macro_corr = X_train[macro_cols].corr()
labels = [c.replace('_Log_Return', '').replace('_', '\n') for c in macro_cols]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(macro_corr, dtype=bool))
sns.heatmap(macro_corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Macro Feature Correlation Matrix (Train Set)', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

---
## 4. Model Training & Hyperparameter Tuning

###  4.1 Baseline Model

Standard random forest model baseline without tuning












In [ ]:
baseline_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
baseline_rf.fit(X_train, y_train)

y_pred_base       = baseline_rf.predict(X_test)
y_pred_base_proba = baseline_rf.predict_proba(X_test)[:, 1]

print('BASELINE: Default Random Forest')
print(f'  Accuracy   : {accuracy_score(y_test, y_pred_base):.4f}')
print(f'  ROC-AUC    : {roc_auc_score(y_test, y_pred_base_proba):.4f}')
print(f'  F1 (Macro) : {f1_score(y_test, y_pred_base, average="macro"):.4f}')
print(f'  Precision  : {precision_score(y_test, y_pred_base):.4f}')
print(f'  Recall     : {recall_score(y_test, y_pred_base):.4f}')
print(f'  Parameters : all sklearn defaults')
print(f'  n_estimators=100, max_depth=None, max_features=sqrt')

### 4.2 Hyperparameter 

We used **RandomizedSearchCV** with **TimeSeriesSplit**


| Parameter | Range | Rationale |
|---|---|---|
| `n_estimators` | 300–1000 | More trees = more stable predictions; diminishing returns past ~500 |
| `max_depth` | 5, 8, 10, 15, 20 | Caps tree depth to prevent overfitting to training regime |
| `min_samples_split` | 10–50 | Minimum samples required to split a node — higher = simpler trees |
| `min_samples_leaf` | 5–20 | Minimum samples at each leaf — primary regularisation lever |
| `max_features` | sqrt, log2, 0.2, 0.3 | Features considered per split; `sqrt` is the RF classification default |
| `max_samples` | 0.5–1.0 | Fraction of training data each tree sees — increases tree diversity |
| `class_weight` | None, balanced | Adjusts for any subtle class imbalance between Up/Down days |


In [ ]:
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

print(
    f'Tuning dataset: {X_trainval.shape[0]} days ({X_train.shape[0]} train + {X_val.shape[0]} val)')

param_dist = {
    'n_estimators':      randint(300, 1001),
    'max_depth':         [3, 5, 8, 10, 15, 20],
    'min_samples_split': randint(10, 51),
    'min_samples_leaf':  randint(5, 21),
    'max_features':      ['sqrt', 'log2', 0.2, 0.3],
    'max_samples':        uniform(0.5, 0.5),
    'class_weight':      [None, 'balanced'],
    'min_weight_fraction_leaf': uniform(0.0, 0.05)
}

tscv = TimeSeriesSplit(n_splits=10)
base_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=200,
    cv=tscv,
    scoring='f1_macro',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_trainval, y_trainval)

print(f'\nBest CV ROC-AUC: {search.best_score_:.4f}')
print(f'Best Parameters:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

### 4.3 Retrain Best Model on Full Train + Val Set

In [ ]:

# define best_rf on all features
best_rf = RandomForestClassifier(**search.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
best_rf.fit(X_trainval, y_trainval)
print('Initial model trained on all features.')

# Now compute permutation importance to identify weak features
print('Computing permutation importance for feature selection')
perm_imp_full = permutation_importance(
    best_rf, X_test, y_test,
    n_repeats=20, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1
)

perm_series = pd.Series(perm_imp_full.importances_mean, index=X_trainval.columns)                                                                   

# Keep only features with positive permutation importance
selected_features = perm_series[perm_series > 0].index.tolist()
print(f'Features kept: {len(selected_features)} / {len(X_trainval.columns)}')

# Refit on reduced feature set
X_trainval_sel = X_trainval[selected_features]
X_test_sel     = X_test[selected_features]

best_rf = RandomForestClassifier(**search.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
best_rf.fit(X_trainval_sel, y_trainval)
print(f'Final model retrained on {len(selected_features)} selected features.')

---
## 5. Evaluation on Test Set (2025)

In [ ]:
y_pred       = best_rf.predict(X_test_sel)
y_pred_proba = best_rf.predict_proba(X_test_sel)[:, 1]

# Core metrics
acc       = accuracy_score(y_test, y_pred)
f1_macro  = f1_score(y_test, y_pred, average='macro')
f1_up     = f1_score(y_test, y_pred, pos_label=1)
f1_down   = f1_score(y_test, y_pred, pos_label=0)
prec      = precision_score(y_test, y_pred)
rec       = recall_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_pred_proba)


print('TEST SET PERFORMANCE (2025)')
print(f'  Accuracy        : {acc:.4f}')
print(f'  ROC-AUC         : {auc:.4f}')
print(f'  F1 (Macro)      : {f1_macro:.4f}')
print(f'  F1 (Up / class1): {f1_up:.4f}')
print(f'  F1 (Down/class0): {f1_down:.4f}')
print(f'  Precision (Up)  : {prec:.4f}')
print(f'  Recall (Up)     : {rec:.4f}')

print('\nFull Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)']))

In [ ]:
from sklearn.metrics import precision_recall_curve

# Find threshold that maximises F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f'Optimal threshold (max F1): {best_thresh:.4f}')

# Re-evaluate at optimal threshold
y_pred_tuned = (y_pred_proba >= best_thresh).astype(int)

print('\n--- Default Threshold (0.50) ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f} | '
      f'F1: {f1_score(y_test, y_pred, average="macro"):.4f} | '
      f'AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

print(f'\n--- Tuned Threshold ({best_thresh:.2f}) ---')
print(f'Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f} | '
      f'F1: {f1_score(y_test, y_pred_tuned, average="macro"):.4f} | '
      f'AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

# Plot Precision-Recall tradeoff
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precisions[:-1], label='Precision', color='#1565C0', lw=2)
ax.plot(thresholds, recalls[:-1],    label='Recall',    color='#43A047', lw=2)
ax.plot(thresholds, f1_scores[:-1],  label='F1',        color='#E53935', lw=2)
ax.axvline(best_thresh, color='grey', ls='--', lw=1.5,
           label=f'Optimal threshold = {best_thresh:.2f}')
ax.axvline(0.5, color='black', ls=':', lw=1, alpha=0.5, label='Default = 0.50')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Classification Threshold', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.show()

### 5.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred Down', 'Pred Up'],
            yticklabels=['True Down', 'True Up'],
            linewidths=1, cbar=False)
axes[0].set_title('Confusion Matrix — Counts', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# Row-normalised percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['Pred Down', 'Pred Up'],
            yticklabels=['True Down', 'True Up'],
            linewidths=1, cbar_kws={'label': '%'})
axes[1].set_title('Confusion Matrix — Row %', fontweight='bold')
axes[1].set_ylabel('Actual'); axes[1].set_xlabel('Predicted')

# Annotate with % symbol
for text in axes[1].texts:
    text.set_text(text.get_text() + '%')

plt.suptitle('XLE Direction Prediction — Test Set 2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#1565C0', lw=2.5, label=f'Random Forest (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5000)', alpha=0.6)
ax.fill_between(fpr, tpr, alpha=0.08, color='#1565C0')


op_idx = np.argmin(np.abs(thresholds - 0.5))
ax.scatter(fpr[op_idx], tpr[op_idx], s=100, color='#E53935',
           zorder=5, label=f'Threshold=0.5 (FPR={fpr[op_idx]:.2f}, TPR={tpr[op_idx]:.2f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — XLE Direction (Test 2025)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

### 5.3 Predicted Probability Distribution Over Time (2025)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

# Top: probability over time coloured by correctness
ax = axes[0]
correct_mask = (y_pred == y_test.values)
dates = X_test.index

ax.axhline(0.5, color='grey', ls='--', lw=1, alpha=0.7, label='Decision boundary (0.5)')
ax.scatter(dates[correct_mask],  y_pred_proba[correct_mask],
           c='#43A047', s=18, alpha=0.8, label='Correct', zorder=3)
ax.scatter(dates[~correct_mask], y_pred_proba[~correct_mask],
           c='#E53935', s=18, alpha=0.8, label='Incorrect', zorder=3)
ax.set_ylabel('P(Up | features)', fontsize=11)
ax.set_title('Predicted Probability of XLE Going Up — Test Set 2025', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# Bottom: histogram of predicted probabilities
ax2 = axes[1]
ax2.hist(y_pred_proba[y_test.values == 1], bins=30, alpha=0.65,
         color='#43A047', label='True Up (1)', density=True)
ax2.hist(y_pred_proba[y_test.values == 0], bins=30, alpha=0.65,
         color='#E53935', label='True Down (0)', density=True)
ax2.axvline(0.5, color='grey', ls='--', lw=1.5)
ax2.set_xlabel('Predicted Probability of Up', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('Distribution of Predicted Probabilities by True Class', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Sensitivity Analysis

### 6.1 Gini Feature Importance (Top 30)

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=selected_features)
top30 = importances.nlargest(30).sort_values()

# Colour macro features differently
macro_names = ['VIX', 'Oil_Volatility', 'Crude_Oil', 'Natural_Gas', 'Dollar', 'SP500', 'Nikkei', 'FTSE']
colors_fi = ['#1565C0' if any(m in f for m in macro_names) else '#78909C' for f in top30.index]

fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(top30.index.str.replace('_Log_Return', '').str.replace('_', ' '),
               top30.values, color=colors_fi, edgecolor='white')

# Value labels
for bar, val in zip(bars, top30.values):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1565C0', label='Macro / Market Feature'),
    Patch(facecolor='#78909C', label='Individual Energy Stock')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

ax.set_xlabel('Mean Decrease in Gini Impurity', fontsize=11)
ax.set_title('Top 30 Feature Importances — Random Forest\n(Gini / MDI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 features by Gini importance:')
print(importances.nlargest(10).to_string())

### 6.2 Permutation Importance on Test Set


In [ ]:
print('Computing permutation importance on test set')
perm_result = permutation_importance(
    best_rf, X_test_sel, y_test,
    n_repeats=20,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_imp = pd.DataFrame({
    'feature':    X_test_sel.columns,
    'importance': perm_result.importances_mean,
    'std':        perm_result.importances_std
}).sort_values('importance', ascending=False)

print('Permutation importance computed.')
print('\nTop 10 features by permutation importance (ROC-AUC drop):')
print(perm_imp.head(10)[['feature','importance','std']].to_string(index=False))

In [ ]:
top30_perm = perm_imp.head(30).sort_values('importance')
colors_perm = ['#1565C0' if any(m in f for m in macro_names) else '#78909C'
               for f in top30_perm['feature']]

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(
    top30_perm['feature'].str.replace('_Log_Return', '').str.replace('_', ' '),
    top30_perm['importance'],
    xerr=top30_perm['std'],
    color=colors_perm,
    capsize=3, edgecolor='white'
)
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_xlabel('Mean ROC-AUC Decrease when Feature is Shuffled', fontsize=11)
ax.set_title('Top 30 Permutation Importances — Test Set 2025\n(error bars = ±1 std over 20 repeats)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Macro vs. Stock Features — Ablation Study

In [ ]:
macro_cols_all = [c for c in X_train.columns if any(m in c for m in macro_names)]
stock_cols_all = [c for c in X_train.columns if c not in macro_cols_all]

ablation_results = {}

for label, cols in [('Macro Only', macro_cols_all),
                     ('Stocks Only', stock_cols_all),
                     ('Full Model', list(selected_features))]:
    rf_ab = RandomForestClassifier(**search.best_params_, random_state=RANDOM_STATE, n_jobs=-1)
    rf_ab.fit(X_trainval[cols], y_trainval)
    preds      = rf_ab.predict(X_test[cols])
    preds_prob = rf_ab.predict_proba(X_test[cols])[:, 1]
    ablation_results[label] = {
        'Accuracy':  accuracy_score(y_test, preds),
        'F1 Macro':  f1_score(y_test, preds, average='macro'),
        'ROC-AUC':   roc_auc_score(y_test, preds_prob),
        'Precision': precision_score(y_test, preds),
        'Recall':    recall_score(y_test, preds),
        'n_features': len(cols)
    }

ablation_df = pd.DataFrame(ablation_results).T
print(ablation_df.round(4).to_string())

In [ ]:
metrics_to_plot = ['Accuracy', 'F1 Macro', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25
group_colors = ['#1565C0', '#78909C', '#E65100']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (label, row) in enumerate(ablation_df.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=label,
                  color=group_colors[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', ls='--', lw=1, alpha=0.5, label='Random baseline')
ax.set_title('Macro Only vs Stocks Only vs Full Model\n(Test Set 2025)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 6.4 Hyperparameter Search — CV Score Distribution

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# CV score distribution
axes[0].hist(cv_results['mean_test_score'], bins=25, color='#1565C0', alpha=0.8, edgecolor='white')
axes[0].axvline(search.best_score_, color='#E53935', ls='--', lw=2,
                label=f'Best: {search.best_score_:.4f}')
axes[0].set_xlabel('Mean CV F1 Macro') 
axes[1].set_ylabel('Mean CV F1 Macro')  
axes[0].set_title('Distribution of CV Scores', fontweight='bold')
axes[0].legend()

# n_estimators vs score
axes[1].scatter(cv_results['param_n_estimators'],
                cv_results['mean_test_score'],
                alpha=0.5, s=30, color='#1565C0')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Mean CV ROC-AUC')
axes[1].set_title('n_estimators vs CV Score', fontweight='bold')

plt.suptitle('RandomizedSearchCV — Hyperparameter Exploration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Summary

In [ ]:
print('=' * 55)
print('           FINAL MODEL SUMMARY')
print('=' * 55)
print(f'  Model         : Random Forest Classifier')
print(f'  Training data : 2016–01–01 to 2023–12–31')
print(f'  Tuning method : RandomizedSearchCV + TimeSeriesSplit (n_splits={tscv.n_splits})')
print(f'  Test period   : 2025–01–01 to 2025–12–31')
print(f'  Features      : {len(selected_features)} selected from {X_train.shape[1]} original (8 macro + {X_train.shape[1]-8} energy stocks)')
print(f'  Random state  : {RANDOM_STATE}')
print()
print('  Best Hyperparameters:')
for k, v in search.best_params_.items():
    print(f'    {k}: {v}')
print()
print('  Test Set Performance:')
print(f'    Accuracy   : {acc:.4f}')
print(f'    ROC-AUC    : {auc:.4f}')
print(f'    F1 (Macro) : {f1_macro:.4f}')
print(f'    Precision  : {prec:.4f}')
print(f'    Recall     : {rec:.4f}')


# Summary of changes and impact to accuracy:

### Baseline Model without hyperparameter: n_splits=5,  n_iter=200, scoring=f1_macro 
    Accuracy   : 0.5100
    ROC-AUC    : 0.4957
    F1 (Macro) : 0.4935
    Precision  : 0.5584
    Recall     : 0.6143

## Hyperperametertuned models:

### 1. n_splits=5,  n_iter=200
    Accuracy   : 0.5502
    ROC-AUC    : 0.5531
    F1 (Macro) : 0.5350
    Precision  : 0.5909
    Recall     : 0.6500


### 2. n_splits=10,  n_iter=200
    Accuracy   : 0.5582
    ROC-AUC    : 0.5844
    F1 (Macro) : 0.5574
    Precision  : 0.6250
    Recall     : 0.5357

### 3. n_splits=5,  n_iter=300
    Accuracy   : 0.5502
    ROC-AUC    : 0.5531
    F1 (Macro) : 0.5350
    Precision  : 0.5909
    Recall     : 0.6500

### 4. n_splits=10,  n_iter=300, gap=5
    Accuracy   : 0.5622
    ROC-AUC    : 0.5486
    F1 (Macro) : 0.5494
    Precision  : 0.6026
    Recall     : 0.6500

### 5. n_splits=10,  n_iter=300, gap=5, 'max_depth': [3, 5, 8, 10, 15, 20] , 
### min_weight_fraction_leaf: uniform(0.0, 0.05)
    
    Accuracy   : 0.5703
    ROC-AUC    : 0.5478
    F1 (Macro) : 0.5589
    Precision  : 0.6107
    Recall     : 0.6500


   

